## ASG Airlines — Data Ingestion & Quality Audit
Loads raw source data and profiles it for quality issues (missing values,
duplicates, invalid formats, referential integrity) before any cleaning
is applied. Output of this notebook is a written audit report used to
drive the cleaning logic in the next notebook.

In [3]:
import os
print(os.path.exists("D:/asg-airlines-pipeline/data/raw/UseCase - Airlines.xlsx"))
print(os.listdir("D:/asg-airlines-pipeline/data/raw"))

True
['UseCase - Airlines.xlsx']


In [5]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd

RAW_PATH = "D:/asg-airlines-pipeline/data/raw/asg_airlines_raw.xlsx"

flights   = pd.read_excel(RAW_PATH, sheet_name="flights")
payments  = pd.read_excel(RAW_PATH, sheet_name="payments")
bookings  = pd.read_excel(RAW_PATH, sheet_name="bookings")
passengers = pd.read_excel(RAW_PATH, sheet_name="passengers")

for name, df in [("flights", flights), ("payments", payments),
                  ("bookings", bookings), ("passengers", passengers)]:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols")

flights: 1020 rows, 7 cols
payments: 1000 rows, 4 cols
bookings: 1000 rows, 9 cols
passengers: 1039 rows, 9 cols


In [10]:
def audit(df, name, key_cols=None):
    print(f"\n===== AUDIT: {name} =====")
    print("Missing values per column:\n", df.isna().sum())
    print("\nFull-row duplicates:", df.duplicated().sum())
    if key_cols:
        for col in key_cols:
            print(f"Duplicate {col} values:", df[col].duplicated().sum())
    print("\nDtypes:\n", df.dtypes)

audit(flights, "flights", key_cols=["flight_id"])
audit(payments, "payments", key_cols=["payment_id"])
audit(bookings, "bookings", key_cols=["booking_id"])
audit(passengers, "passengers", key_cols=["passenger_id"])


===== AUDIT: flights =====
Missing values per column:
 flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

Full-row duplicates: 15
Duplicate flight_id values: 16

Dtypes:
 flight_id                    str
airline                      str
source                       str
destination                  str
departure_time    datetime64[us]
arrival_time      datetime64[us]
duration                  object
dtype: object

===== AUDIT: payments =====
Missing values per column:
 payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64

Full-row duplicates: 0
Duplicate payment_id values: 0

Dtypes:
 payment_id           str
booking_id           str
amount            object
payment_method       str
dtype: object

===== AUDIT: bookings =====
Missing values per column:
 booking_id                  0
passenger_id                0
flight_id              

In [ ]:

overnight = flights[flights["arrival_time"].dt.date > flights["departure_time"].dt.date]
print("Overnight flights:", len(overnight))


bad_time_order = flights[flights["arrival_time"] < flights["departure_time"]]
print("Arrival-before-departure errors:", len(bad_time_order))
print(bad_time_order[["flight_id","departure_time","arrival_time"]])


print("\nAirline value counts:\n", flights["airline"].value_counts(dropna=False))


non_numeric_amount = payments[pd.to_numeric(payments["amount"], errors="coerce").isna()
                               & payments["amount"].notna()]
print("\nNon-numeric payment amounts:", len(non_numeric_amount))
print(non_numeric_amount["amount"].value_counts())


print("\nBooking status values:\n", bookings["status"].value_counts(dropna=False))


orphan_bookings = bookings[~bookings["flight_id"].isin(flights["flight_id"])]
orphan_payments = payments[~payments["booking_id"].isin(bookings["booking_id"])]
orphan_passengers_in_bookings = bookings[~bookings["passenger_id"].isin(passengers["passenger_id"])]
print("\nOrphan bookings (bad flight_id):", len(orphan_bookings))
print("Orphan payments (bad booking_id):", len(orphan_payments))
print("Orphan bookings (bad passenger_id):", len(orphan_passengers_in_bookings))

Overnight flights: 124
Arrival-before-departure errors: 1
    flight_id      departure_time        arrival_time
355     SJ192 2026-04-19 18:45:42 2026-04-18 23:45:42

Airline value counts:
 airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           41
UNKNOWN       31
Name: count, dtype: int64

Non-numeric payment amounts: 30
amount
INVALID    30
Name: count, dtype: int64

Booking status values:
 status
CONFIRMED    320
CANCELLED    314
PENDING      291
NaN           45
INVALID       30
Name: count, dtype: int64

Orphan bookings (bad flight_id): 0
Orphan payments (bad booking_id): 0
Orphan bookings (bad passenger_id): 0
